In [ ]:
import anthropic
from pathlib import Path
import csv
import pandas as pd
from dotenv import load_dotenv
import re

Load the Anthropic API from .env file

In [3]:
load_dotenv("../etc/.env")  # loads .env into environment variables

True

In [2]:
client = anthropic.Anthropic()

Load data from csv file, containing statements.

In [56]:
df = pd.read_csv("exercises_conf_electronica.csv")

In [57]:
df.head()

,subject,topic,year,exam,exercise,statement
0,quiímica,Conf. Electrónica,2000,Junio,Ejercicio 2 Opción B,Los números atómicos de los elementos P y Mn s...
1,química,Conf. Electrónica,2000,Reserva 1,Ejercicio 4 Opcion A,"Los elementos Na, Al, y Cl tienen números atóm..."
2,quimica,Conf. Electrónica,2000,Reserva 2,Ejercicio 2 Opcion A,"Tres elementos tienen de número atómico 25, 35..."
3,quimica,Conf. Electrónica,2000,Reserva 3,Ejercicio 2 Opción A,"Los elementos A y B tienen, en sus últimos niv..."
4,quimica,Conf. Electrónica,2000,Reserva 4,Ejercicio 2 Opción B,Los números atómicos de los elementos Br y Rb ...


Replace new line character with "|" in statement, to avoid problems with csv

In [58]:
df.statement = df.statement.str.replace("\n", " | ")

Divide the dataset into pieces so the AI model can process it within the tokens limit.

In [60]:
df_chunks = [df[i:i+50] for i in range(0, len(df), 50) ]
print(len(df_chunks))

4


Function to process a csv file and classify them using AI

In [ ]:
def clasificar_ejercicios(dataset: pd.DataFrame) -> list[str]:
    
    system_prompt = Path(f"{topic}.md").read_text()
    dataset.to_csv("data_csv.csv", index=False)
        
    data_text = Path("data_csv.csv").read_text()        # subset, no df completo

    # remove the data_csv.csv file after reading its content
    Path("data_csv.csv").unlink(missing_ok=True)

    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=8192,
        system=system_prompt,
        messages=[{"role": "user",
                    "content": f"Here is the data: \n\n{data_text}"
                    }]
    )

    return response.content[0].text.strip().splitlines()

The response includes the problem statements, which cause parsing issues when converting to CSV. I'm removing the problem statements.

In [ ]:
# def remove_statement(line):
#     start, end = [i.start() for i in re.finditer('"', line)]
#     line = line[:start-1] + line[end+1:] 
#     return line

Process the different chunks of the exercises csv file.

In [ ]:
response = clasificar_ejercicios(df_chunks[4])

Clean the response

In [83]:
response[:10]

['```csv',
 'subject,topic,year,exam,exercise,tipo_ejercicio',
 'química,Conf. Electrónica,2022,Junio,Ejercicio B1,Combinaciones de números cuánticos | Situar elementos en tabla periódica (grupo y período)',
 'quiímica,Conf. Electrónica,2022,Reserva 1,Ejercicio B1,Ion más estable | Situar elementos en tabla periódica (grupo y período)',
 'quiímica,Conf. Electrónica,2022,Reserva 1,Ejercicio B2,Propiedades periódicas',
 'quimica,Conf. Electrónica,2022,Reserva 2,Ejercicio B1,Verdadero o Falso',
 'quiímica,Conf. Electrónica,2022,Reserva 2,Ejercicio B4,Comparación energías de ionización | Situar elementos en tabla periódica (grupo y período)',
 'quimica,Conf. Electrónica,2022,Reserva 3,Ejercicio B1,Combinaciones de números cuánticos',
 'quimica,Conf. Electrónica,2022,Reserva 3,Ejercicio B4,Propiedades periódicas | Situar elementos en tabla periódica (grupo y período)',
 'quimica,Conf. Electrónica,2022,Reserva 4,Ejercicio B1,Verdadero o Falso']

In [84]:
response = response[2:]

In [85]:
response[:5]

['química,Conf. Electrónica,2022,Junio,Ejercicio B1,Combinaciones de números cuánticos | Situar elementos en tabla periódica (grupo y período)',
 'quiímica,Conf. Electrónica,2022,Reserva 1,Ejercicio B1,Ion más estable | Situar elementos en tabla periódica (grupo y período)',
 'quiímica,Conf. Electrónica,2022,Reserva 1,Ejercicio B2,Propiedades periódicas',
 'quimica,Conf. Electrónica,2022,Reserva 2,Ejercicio B1,Verdadero o Falso',
 'quiímica,Conf. Electrónica,2022,Reserva 2,Ejercicio B4,Comparación energías de ionización | Situar elementos en tabla periódica (grupo y período)']

Convert the response into a string

In [86]:
classified_data = "\n".join(response)

Write the string into a csv file

In [87]:
with open("classified_conf_electronica.csv", "a", encoding="utf-8") as file:
    file.writelines(classified_data)
    file.write("\n")  # Add a newline at the end of the file

Load the generated csv file to check the results

In [89]:
df2 = pd.read_csv("classified_conf_electronica.csv", encoding="utf-8")

In [90]:
df2.subject = "química"

In [91]:
df2.sample(10)

,subject,topic,year,exam,exercise,exercise_type
175,química,Conf. Electrónica,2025,Junio,Ejercicio 1A,Propiedades periódicas
10,química,Conf. Electrónica,2001,Reserva 4,Ejercicio 2 Opcion A,Combinaciones de números cuánticos
31,química,Conf. Electrónica,2005,Reserva 1,Ejercicio 3 Opción B,Radio / energía ionización Na+ y Ne
126,química,Conf. Electrónica,2019,Reserva 4,Ejercicio 2 Opción B,Situar elementos en tabla periódica (grupo y p...
92,química,Conf. Electrónica,2015,Reserva 2,Ejercicio 2 Opción A,Verdadero o Falso | Propiedades periódicas
168,química,Conf. Electrónica,2024,Reserva 2,Ejercicio B2,Propiedades periódicas
5,química,Conf. Electrónica,2000,Septiembre,Ejercicio 2 Opción B,Propiedades periódicas
44,química,Conf. Electrónica,2007,Reserva 2,Ejercicio 2 Opción B,Verdadero o Falso
16,química,Conf. Electrónica,2002,Reserva 4,Ejercicio 2 Opción A,Situar elementos en tabla periódica (grupo y p...
155,química,Conf. Electrónica,2022,Reserva 3,Ejercicio B1,Combinaciones de números cuánticos


Sort the dataset by year, exam and exercise

In [94]:
df2 = df2.sort_values(["year", "exam", "exercise"]).reset_index(drop = True)

Export to csv file the sorted dataframe

In [95]:
df2.to_csv("classified_conf_electronica.csv", index = False)